# M3 primary and cosmetics preparation
Both sources are enabled and remain separate. This notebook reviews actual M3 results; it does not run M1 integration tests or train models. Cosmetics unknown dates are not zeros.

In [1]:
from pathlib import Path
import sys,json,csv
ROOT=Path.cwd()
if ROOT.name=="notebooks": ROOT=ROOT.parent
sys.path[:0]=[str(ROOT),str(ROOT/"src")]
from scripts.select_categories import load_config
cfg,_=load_config(ROOT)
with (ROOT/"reports/tables/quality_report.csv").open(encoding="utf-8-sig") as f:
    quality=list(csv.DictReader(f))
for source in ["multi_store","cosmetics"]:
    print(source,{r["metric"]:r["value"] for r in quality if r["dataset"]==source and r["metric"] in ["rows","distinct_category_ids","selected_unresolved_category_days"]})

multi_store {'rows': '36555', 'distinct_category_ids': '691'}
cosmetics {'rows': '63546', 'distinct_category_ids': '525'}


## Separate category proposals and readiness
Selections depend only on their own development periods. Later missingness must not change the selected categories. Cosmetics daily export is withheld until its integer contract can be met.

In [2]:
manifest=json.loads((ROOT/"reports/m3_run_manifest.json").read_text(encoding="utf-8"))
for source,profile in manifest["datasets"].items():
    for count,variant in profile["variants"].items():
        print(source,count,variant["release_status"],variant["output_file"],variant["observed_selected_rows"],variant["unresolved_gaps"])
        print("Candidate IDs:",variant["selected_ids"])


multi_store 10 complete_observed_candidate_not_final_selection data/processed/daily_category.csv 610 0
Candidate IDs: ['2053013555631882655', '2053013553559896355', '2053013554658804075', '2053013554415534427', '2053013563810775923', '2053013558920217191', '2053013566243472379', '2053013556110033341', '2053013555036291451', '2053013565362668491']
multi_store 20 complete_observed_candidate_not_final_selection data/processed/daily_category_top20.csv 1220 0
Candidate IDs: ['2053013555631882655', '2053013553559896355', '2053013554658804075', '2053013554415534427', '2053013563810775923', '2053013558920217191', '2053013553341792533', '2053013565983425517', '2053013563651392361', '2053013553375346967', '2053013563911439225', '2053013563173241677', '2053013566243472379', '2053013556110033341', '2053013555036291451', '2053013553140465927', '2053013557343158789', '2053013553316626707', '2053013565362668491', '2116907504691576833']
cosmetics 10 withheld_unresolved_gaps_no_daily_contract_output No

## Window feasibility after unknown-day exclusion
These are M3 date audits, not actual feature files or model results. The affected cosmetics category has no usable evaluation windows; its score is unavailable.

In [3]:
for source,profile in manifest["datasets"].items():
    for count,variant in profile["variants"].items():
        print(source,count,"unknown dates:",{cid:len(days) for cid,days in variant["unknown_days"].items()})
        print([{k:v for k,v in row.items() if k!="per_category"} for row in variant["split_feasibility"]])


multi_store 10 unknown dates: {}
[{'lookback_days': 7, 'train_per_complete_category': 17, 'validation_per_complete_category': 1, 'test_per_complete_category': 2, 'usable_train_windows': 170, 'usable_validation_windows': 10, 'usable_test_windows': 20, 'excluded_windows': 280, 'status': 'calendar_feasible_not_model_experiment'}, {'lookback_days': 14, 'train_per_complete_category': 10, 'validation_per_complete_category': 1, 'test_per_complete_category': 2, 'usable_train_windows': 100, 'usable_validation_windows': 10, 'usable_test_windows': 20, 'excluded_windows': 280, 'status': 'calendar_feasible_not_model_experiment'}, {'lookback_days': 28, 'train_per_complete_category': 0, 'validation_per_complete_category': 1, 'test_per_complete_category': 2, 'usable_train_windows': 0, 'usable_validation_windows': 10, 'usable_test_windows': 20, 'excluded_windows': 240, 'status': 'blocked_no_training_windows'}]
multi_store 20 unknown dates: {}
[{'lookback_days': 7, 'train_per_complete_category': 17, 'va